# Exploratory Data Analysis

Author: Justin Winkler
Date: September 17, 2026

The purpose of this notebook is to explore the Zillow and data center location datasets.

Effective exploration will satisfy the following criteria:

1. *Profiles* the data:
    - What size is the data?
    - What type(s) does each column contain?
    - What do the distributions of each variable look like?
2. Evaluates the *completeness* of the data:
    - Are there any missing values in the dataset? If so:
        - How many and where?
        - Why might those values be missing?
        - Based on the answer to the following question, what is the best way to handle the missing values?
3. Ensures the *accuracy* of the data:
    - Is the data consistent with other trusted sources?
4. Verifies the *consistency* of data:
    - Are similar measurements recorded in consistent units?
    - Are observations duplicated across datasets? If so, do they match?
5. Enforces the *integrity* of the data:
    - Are IDs unique? Are they consistent between datasets?
6. Documents the data's *lineage and provenance*:
    - Where did the data come from?
    - How has the data been transformed?

## **Import essential data processing utilities**

In [84]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## **Load the data**

First, define a local path to the `data` directory.

In [85]:
from pathlib import Path

data_dir = Path("data").resolve()

#### Load data center location information

Rows in `data_centers.csv` correspond to unique data center locations in the U.S. and abroad. The full dataset contains numerical fields describing the energy consumption and output of each data center, categorial fields containing tags that describe each data center's user and owner, and other data outside the scope of this project.

First, limit data centers to only US locations to match the Zillow dataset. Then, select only the columns pertaining to data center names (a unique identifier for this dataset) and their addresses.

In [86]:
raw_data_centers = pd.read_csv(data_dir / "raw" / "epoch_ai" / "data_centers.csv")
us_data_centers = raw_data_centers[raw_data_centers["Country"] == "United States"]
us_data_center_locations = us_data_centers[["Name", "Address"]]

#### Load data center construction timeline information

Rows in `data_center_timelines.csv` correspond to aggregations of data center construction update news headlines. Fields in this dataset include the name of the data center the construction update belongs to, a description of the update, a count of the number of operational buildings at the data center site at the time of the update, as well as information about the energy consumption and cost of the data center.

Select only the columns pertaining to data center names (a unique identifier for this dataset), the date of the headline aggregation, and the number of buildings that were operational at the time of the aggregation. Rename the "Data center" field to "Name" and the "Buildings operational" field to "BuildingsOperational" to standardize the naming scheme for columns. Standardize date-like fields by converting them to datetime.

In [87]:
data_center_timelines = pd.read_csv(data_dir / "raw" / "epoch_ai" / "data_center_timelines.csv")
data_center_timelines = data_center_timelines[["Data center", "Date", "Buildings operational"]]
data_center_timelines = data_center_timelines.rename(columns={'Data center': 'Name', "Buildings operational": "BuildingsOperational"})
data_center_timelines["Date"] = pd.to_datetime(data_center_timelines.Date)

#### Define the event of interest

Control whether to study the impact of the "first headline" about data center construction or the completion of the first data center's construction on surrounding home values.

In [88]:
from enum import Enum

class EventOfInterest(Enum):
    FIRST_HEADLINE = 1
    FIRST_OPERATIONAL = 2

EVENT_OF_INTEREST = EventOfInterest.FIRST_OPERATIONAL

if EVENT_OF_INTEREST == EventOfInterest.FIRST_OPERATIONAL:
    # If looking for first operational date, remove dates with no operational buildings
    data_center_timelines = data_center_timelines[data_center_timelines["BuildingsOperational"] != 0]

data_center_dates_of_interest = data_center_timelines.groupby(by="Name")["Date"].min()
data_center_dates_of_interest.name = "DateOfInterest (DOI)"
print(data_center_dates_of_interest)

Name
AWS Berwick                      2025-09-09
AWS New Albany                   2024-12-01
Alibaba Zhangbei                 2025-09-17
Amazon Madison Mega Site         2025-06-23
Amazon Ridgeland                 2026-05-15
                                    ...    
Start Campus Sines Data Campus   2026-04-03
Stream Phoenix                   2026-05-01
VNET Bayin Ulanqab               2025-07-01
Vantage TX1                      2026-02-15
xAI QTS Atlanta                  2025-02-20
Name: DateOfInterest (DOI), Length: 86, dtype: datetime64[ns]


#### Combine the data center datasets

Create a unified dataset by joining the data center DataFrames on their name and keeping all columns of the trimmed datasets.

Print the first five entries of the dataset for a preview of the data's structure.

In [89]:
# Join the two primary datasets
data_centers = pd.merge(us_data_center_locations, data_center_dates_of_interest, on="Name", how="outer")

# Extract the last 5-digit run of characters (preceded by whitespace) in the "Address" field to the "ZipCode" field.
data_centers["ZipCode"] = data_centers['Address'].str.extract(r'(?<=\s)(\d{5})(?!.*\d)')

# Manually map a few data centers to their zip, sourced using Google Maps.
data_centers["ZipCode"] = data_centers["ZipCode"].fillna(data_centers["Name"].map({
    "AWS New Albany": "43054",
    "Google The Dalles": "97058",
    "Meta Huntsville": "35810",
    "CoreWeave Chester VA": "23836",
    "Microsoft-Nebius New Jersey": "08361",
    "Stream Phoenix": "85338",
    "Amazon Ridgeland": "39157",
    "Amazon Madison Mega Site": "39046",
}))

# Drop any remaining rows with NaN zip codes
data_centers = data_centers.dropna(subset=['ZipCode'])

# If a ZipCode has multiple dates of interest because more than one data center belongs to it, choose the earliest
data_centers = (
    data_centers.sort_values("DateOfInterest (DOI)")
    .groupby("ZipCode", as_index=False)
    .agg({"Name": list, "Address": "first", "DateOfInterest (DOI)": "first"})
)

# Visualize the new data
print(data_centers.head())

  ZipCode                                           Name  \
0   08361                  [Microsoft-Nebius New Jersey]   
1   14012  [Core42 Lake Mariner, Anthropic Lake Mariner]   
2   18603                                  [AWS Berwick]   
3   20109                   [STACK Infrastructure NVA02]   
4   20136                               [Google Bristow]   

                                      Address DateOfInterest (DOI)  
0    3963 S Lincoln Ave, Vineland, New Jersey           2026-04-15  
1              7725 Lake Rd, Barker, NY 14012           2025-10-01  
2        1125 Electron Ave, Berwick, PA 18603           2025-09-09  
3       9590 Hornbaker Rd, Manassas, VA 20109           2024-10-01  
4  13001 Rollins Ford Road, Bristow, VA 20136           2024-04-29  


#### Load Zillow home estimates

Rows in this dataset correspond to unique U.S. zip codes. The first few columns of the dataset are categorical, specifying the city, county, state, and metro area that the zip code belongs to while the remainder of the columns are numeric and specify the average single-family residence (SFR) value estimate, if one exists, for each month in the period from January 2000 to July 2026.

Select only the columns pertaining to zip codes and monthly average SFR values. Rename the "RegionName" field to "ZipCode". Standardize date-like column names by converting them to datetime.

Print the first five entries of the dataset for a preview of the data's structure.

In [90]:
from datetime import datetime

# Read then concatenate the two zestimate datasets. There are two datasets to accomodate
# GitHub's file size limit.
zestimates1 = pd.read_csv(data_dir / "raw" / "zillow" / "zestimates_by_zip_1.csv")
zestimates2 = pd.read_csv(data_dir / "raw" / "zillow" / "zestimates_by_zip_2.csv")
zestimates = pd.concat([zestimates1, zestimates2], ignore_index=True)

# Split the data into two DataFrame objects:
#   - `zip_geo`: categorical features describing geographical information about zip codes
#   - `zestimates_by_zip`: a chronological breakdown of average home value zestimates by zip code
zestimates_by_zip = zestimates.filter(regex=r'RegionName|\d{4}-\d{2}-\d{2}')
zip_geo = zestimates.filter(regex=r'RegionName|State|Metro|CountyName|SizeRank')

# Rename "RegionName" columns to "ZipCode" and standardize zip codes to 5-character strings.
# Pad the front with 0s, if necessary, since they are originally stored as int64s.
zestimates_by_zip = zestimates_by_zip.rename(columns={"RegionName": "ZipCode"})
zip_geo = zip_geo.rename(columns={"RegionName": "ZipCode"})
zip_geo["ZipCode"] = zip_geo["ZipCode"].astype(str).str.zfill(5)

# Standardize date-like column names
zestimates_by_zip.columns = ["ZipCode", *pd.to_datetime(zestimates_by_zip.columns[1:])]

# Visualize the new data
print(zestimates_by_zip.head())

   ZipCode  2000-01-31 00:00:00  2000-02-29 00:00:00  2000-03-31 00:00:00  \
0    77494        212188.981819        212373.142689        212865.242558   
1     8701        113571.789919        114040.532783        114357.352733   
2    77449        105369.930306        105384.490172        105255.149979   
3    11368        171194.526575        172896.026224        174174.055641   
4    77084        105256.097011        105211.953674        105025.045819   

   2000-04-30 00:00:00  2000-05-31 00:00:00  2000-06-30 00:00:00  \
0        213859.178025        213894.730929        213739.535344   
1        115159.988256        115995.021610        116974.287103   
2        105245.255125        105293.299359        105488.782188   
3        176307.350060        177837.295188        179501.272127   
4        104939.654419        104914.252908        105054.918073   

   2000-07-31 00:00:00  2000-08-31 00:00:00  2000-09-30 00:00:00  ...  \
0        212973.219993        213006.239062        2127

#### Combine data center timeline and home value datasets

In [91]:
# DISCLAIMER: Parts of the following section were generated by Claude Code's Opus 5 model.

# -------------------------- GENERATED WITH HELP FROM CLAUDE CODE --------------------------

EVENT_WINDOW_MONTHS = 24

# Reshape the wide Zillow table into one row per zip code per month. Zillow stores zip
# codes as integers, so zero-pad them to match the five-character strings in data_centers.
zestimates_by_month = zestimates_by_zip.melt(id_vars="ZipCode", var_name="Date", value_name="Zestimate")
zestimates_by_month = zestimates_by_month.dropna(subset="Zestimate")
zestimates_by_month["ZipCode"] = zestimates_by_month["ZipCode"].astype(str).str.zfill(5)
zestimates_by_month["Date"] = pd.to_datetime(zestimates_by_month["Date"])
zestimates_by_month["LogValue"] = np.log(zestimates_by_month["Zestimate"])
zestimates_by_month = zestimates_by_month.sort_values("Date")

# Anchor each treated zip code to the last Zillow observation strictly before its date of
# interest. merge_asof requires both frames to be sorted on the matched date.
anchors = pd.merge_asof(
    data_centers[["ZipCode", "DateOfInterest (DOI)"]].sort_values("DateOfInterest (DOI)"),
    zestimates_by_month[["ZipCode", "Date"]],
    left_on="DateOfInterest (DOI)",
    right_on="Date",
    by="ZipCode",
    direction="backward",
    allow_exact_matches=False,
)
anchors = anchors.rename(columns={"Date": "AnchorDate"}).dropna(subset="AnchorDate")

# Restrict the panel to treated zip codes, express each observation's date in months
# relative to its anchor (EventTime = 0), and re-base log values to the anchor month.
treated_panel = zestimates_by_month.merge(anchors[["ZipCode", "AnchorDate"]], on="ZipCode")
treated_panel["EventTime"] = (
    12 * (treated_panel["Date"].dt.year - treated_panel["AnchorDate"].dt.year)
    + (treated_panel["Date"].dt.month - treated_panel["AnchorDate"].dt.month)
)
treated_panel = treated_panel[treated_panel["EventTime"].abs() <= EVENT_WINDOW_MONTHS].copy()
baseline = treated_panel["LogValue"].where(treated_panel["EventTime"] == 0)
treated_panel["RelLogValue"] = treated_panel["LogValue"] - baseline.groupby(treated_panel["ZipCode"]).transform("first")

# -------------------------- GENERATED WITH HELP FROM CLAUDE CODE --------------------------

treated_panel.to_csv(data_dir / "clean"  / "data_center_zipcode_value_estimates.csv")

In [92]:
treated_zips = treated_panel["ZipCode"].unique().tolist()
treated_zip_geo = zip_geo[zip_geo["ZipCode"].isin(treated_zips)]
untreated_zip_geo = zip_geo[~zip_geo["ZipCode"].isin(treated_zips)]

# Donors come from the same metro, or the same state for treated zips without one.
has_metro = treated_zip_geo["Metro"].notna()
donor_pools = pd.concat([
    treated_zip_geo[has_metro].merge(untreated_zip_geo, on="Metro", suffixes=("", "Donor")),
    treated_zip_geo[~has_metro].merge(untreated_zip_geo, on="State", suffixes=("", "Donor")),
])[["ZipCode", "ZipCodeDonor"]].rename(columns={"ZipCode": "TreatedZip", "ZipCodeDonor": "DonorZip"})

def features_at_anchor(pairs, zip_col):
    # Log price level at the anchor month, and its change over the preceding window.
    pairs = pairs.copy()
    pairs["PreDate"] = pairs["AnchorDate"] - pd.DateOffset(months=EVENT_WINDOW_MONTHS) + pd.offsets.MonthEnd(0)
    values = zestimates_by_month[["ZipCode", "Date", "LogValue"]]
    pairs = pairs.merge(values.rename(columns={"ZipCode": zip_col, "Date": "AnchorDate", "LogValue": "Level"}), on=[zip_col, "AnchorDate"], how="left")
    pairs = pairs.merge(values.rename(columns={"ZipCode": zip_col, "Date": "PreDate", "LogValue": "PreLevel"}), on=[zip_col, "PreDate"], how="left")
    pairs["PreTrend"] = pairs["Level"] - pairs["PreLevel"]
    return pairs.drop(columns=["PreDate", "PreLevel"])

treated_anchors = anchors.rename(columns={"ZipCode": "TreatedZip"})[["TreatedZip", "AnchorDate"]]
donor_features = features_at_anchor(donor_pools.merge(treated_anchors, on="TreatedZip"), "DonorZip")
donor_features = donor_features.dropna(subset=["Level", "PreTrend"])
treated_features = features_at_anchor(treated_anchors, "TreatedZip")

FEATURES = ["Level", "PreTrend"]
pool_stats = donor_features.groupby("TreatedZip")[FEATURES].agg(["mean", "std"])
pool_stats.columns = [f"{feature}{stat.title()}" for feature, stat in pool_stats.columns]  # LevelMean, LevelStd, ...

def z_score(frame):
    # Standardize each feature against the treated zip's own donor pool.
    frame = frame.merge(pool_stats, left_on="TreatedZip", right_index=True)
    for feature in FEATURES:
        frame[f"{feature}Z"] = (frame[feature] - frame[f"{feature}Mean"]) / frame[f"{feature}Std"]
    return frame.drop(columns=pool_stats.columns)

donor_features = z_score(donor_features)
treated_features = z_score(treated_features)

print(donor_features)
print(treated_features)

     TreatedZip DonorZip AnchorDate      Level  PreTrend    LevelZ  PreTrendZ
0         78245    78130 2024-04-30  12.717869 -0.059314  0.195363  -0.802218
1         78245    78254 2024-04-30  12.694247 -0.006028  0.138807   0.200880
2         78245    78249 2024-04-30  12.674951  0.040621  0.092611   1.079027
3         78245    78253 2024-04-30  12.736743 -0.030064  0.240551  -0.251597
4         78245    78250 2024-04-30  12.449095  0.019207 -0.448126   0.675923
...         ...      ...        ...        ...       ...       ...        ...
6467      58436    58011 2025-09-30  12.422204 -0.023875  0.218029  -0.638104
6469      58436    58793 2025-09-30  11.763382 -0.212127 -1.657565  -2.890196
6470      58436    58736 2025-09-30  11.417819 -0.139287 -2.641344  -2.018802
6472      58436    58655 2025-09-30  12.763259  0.104727  1.188975   0.900397
6474      58436    58565 2025-09-30  12.204913  0.162231 -0.400573   1.588330

[6428 rows x 7 columns]
   TreatedZip AnchorDate      Level  Pr

In [93]:
K_NEAREST = 5

# Euclidean distance from each donor to its treated zip in standardized feature space,
# then keep the K_NEAREST closest donors per treated zip.
z_columns = [f"{feature}Z" for feature in FEATURES]
candidates = donor_features.merge(
    treated_features[["TreatedZip", *z_columns]], on="TreatedZip", suffixes=("", "Treated")
)
candidates["Distance"] = np.sqrt(sum(
    (candidates[column] - candidates[f"{column}Treated"]) ** 2 for column in z_columns
))
candidates = candidates.sort_values(["TreatedZip", "Distance"])
candidates["Rank"] = candidates.groupby("TreatedZip").cumcount() + 1
matched_donors = (
    candidates[candidates["Rank"] <= K_NEAREST][["TreatedZip", "DonorZip", "Rank", "Distance"]]
    .reset_index(drop=True)
)

# Every treated zip should have exactly K_NEAREST donors.
donors_per_zip = matched_donors.groupby("TreatedZip").size()
assert (donors_per_zip == K_NEAREST).all(), donors_per_zip[donors_per_zip != K_NEAREST]

# Put each treated zip's raw features beside its donors' so the match quality can be eyeballed.
match_review = pd.concat([
    treated_features.assign(DonorZip="(treated)", Rank=0, Distance=0.0),
    matched_donors.merge(donor_features, on=["TreatedZip", "DonorZip"]),
])[["TreatedZip", "DonorZip", "Rank", "Distance", "Level", "PreTrend"]].sort_values(["TreatedZip", "Rank"])
match_review["Zestimate"] = np.exp(match_review["Level"]).round(-3)
match_review["PreTrendPct"] = (100 * (np.exp(match_review["PreTrend"]) - 1)).round(1)

# Glance at the best- and worst-matched treated zips beside their donors.
worst_distance = matched_donors.groupby("TreatedZip")["Distance"].max().sort_values()
for treated_zip in [worst_distance.index[0], worst_distance.index[-1]]:
    print(f"\nTreatedZip {treated_zip}")
    print(match_review[match_review["TreatedZip"] == treated_zip].drop(columns="TreatedZip").to_string(index=False))


TreatedZip 28905
 DonorZip  Rank  Distance     Level  PreTrend  Zestimate  PreTrendPct
(treated)     0  0.000000 12.381928  0.044790   238000.0          4.6
    28513     1  0.052392 12.357796  0.043439   233000.0          4.4
    28612     2  0.054405 12.363118  0.047110   234000.0          4.8
    27932     3  0.055699 12.356759  0.043251   233000.0          4.4
    28314     4  0.075916 12.343107  0.045329   229000.0          4.6
    28390     5  0.086231 12.422466  0.042759   248000.0          4.4

TreatedZip 18603
 DonorZip  Rank  Distance     Level  PreTrend  Zestimate  PreTrendPct
(treated)     0  0.000000 12.266490  0.023123   212000.0          2.3
    17985     1  1.865798 12.310537  0.105549   222000.0         11.1
    17814     2  1.921887 12.401940 -0.002657   243000.0         -0.3
    17820     3  2.097996 12.412347 -0.008949   246000.0         -0.9
    17815     4  2.352718 12.439096  0.014602   252000.0          1.5
    17821     5  3.059032 12.489648  0.041841   266000

In [94]:
# Build each matched donor's path in its treated zip's event time. A donor's clock is set
# by the treated zip's anchor, and its log values are re-based to its own value at that
# month, so every (TreatedZip, DonorZip) pair passes through 0 at EventTime 0.
donor_values = zestimates_by_month[zestimates_by_month["ZipCode"].isin(matched_donors["DonorZip"])]
donor_panel = (
    matched_donors.merge(treated_anchors, on="TreatedZip")
    .merge(donor_values.rename(columns={"ZipCode": "DonorZip"}), on="DonorZip")
)
donor_panel["EventTime"] = (
    12 * (donor_panel["Date"].dt.year - donor_panel["AnchorDate"].dt.year)
    + (donor_panel["Date"].dt.month - donor_panel["AnchorDate"].dt.month)
)
donor_panel = donor_panel[donor_panel["EventTime"].abs() <= EVENT_WINDOW_MONTHS].copy()
baseline = donor_panel["LogValue"].where(donor_panel["EventTime"] == 0)
donor_panel["RelLogValue"] = donor_panel["LogValue"] - baseline.groupby(
    [donor_panel["TreatedZip"], donor_panel["DonorZip"]]
).transform("first")

# Average the K_NEAREST donor paths into one synthetic control path per treated zip.
control_panel = (
    donor_panel.groupby(["TreatedZip", "EventTime"], as_index=False)
    .agg(RelLogValue=("RelLogValue", "mean"), DonorCount=("DonorZip", "size"))
)

# Every treated zip's control path should pass through 0 at EventTime 0, and every point
# on it should be built from all K_NEAREST donors.
at_anchor = control_panel[control_panel["EventTime"] == 0]
assert len(at_anchor) == treated_anchors["TreatedZip"].nunique(), "a treated zip has no control path"
assert np.allclose(at_anchor["RelLogValue"], 0), "control path is not re-based to EventTime 0"
short_handed = control_panel[control_panel["DonorCount"] < K_NEAREST]
print(f"{control_panel['TreatedZip'].nunique()} control paths, {len(control_panel)} (TreatedZip, EventTime) points")
print(f"{len(short_handed)} points built from fewer than {K_NEAREST} donors")
print(control_panel.groupby("EventTime")["TreatedZip"].size().rename("TreatedZipsObserved").iloc[::6].to_string())

55 control paths, 1990 (TreatedZip, EventTime) points
0 points built from fewer than 5 donors
EventTime
-24    55
-18    55
-12    55
-6     55
 0     55
 6     31
 12    23
 18    18
 24    15


In [ ]:
# Join each treated zip's own path to its synthetic control path and take the difference.
# Gap is the treated zip's log appreciation since the anchor, net of what its matched donors
# did over the same months; exp(Gap) - 1 is the same thing in percent.
gap_panel = (
    treated_panel[["ZipCode", "EventTime", "RelLogValue"]]
    .rename(columns={"ZipCode": "TreatedZip", "RelLogValue": "Treated"})
    .merge(control_panel.rename(columns={"RelLogValue": "Control"}), on=["TreatedZip", "EventTime"])
)
gap_panel["Gap"] = gap_panel["Treated"] - gap_panel["Control"]

# Nothing should be lost in the join, and the gap is 0 at the anchor by construction.
assert len(gap_panel) == len(treated_panel), "treated and control panels do not line up"
assert np.allclose(gap_panel.loc[gap_panel["EventTime"] == 0, "Gap"], 0)
assert gap_panel["Gap"].notna().all()

print(gap_panel.head(10).to_string(index=False))

In [ ]:
N_BOOTSTRAP = 500
RANDOM_SEED = 510

# Point estimate at each event time: the mean gap across treated zips observed there, with
# the mean treated and control paths for plotting and the count behind each point.
event_study = gap_panel.groupby("EventTime").agg(
    Treated=("Treated", "mean"), Control=("Control", "mean"), Gap=("Gap", "mean"), N=("TreatedZip", "size")
)

# Bootstrap the interval by resampling treated zips, not rows. Months within a zip are
# serially correlated, so the zip is the unit of independent variation. Each draw takes a
# whole zip's path along with it; unobserved months are NaN and drop out of the mean.
rng = np.random.default_rng(RANDOM_SEED)
gap_by_zip = gap_panel.pivot(index="TreatedZip", columns="EventTime", values="Gap")
bootstrap_means = np.empty((N_BOOTSTRAP, gap_by_zip.shape[1]))
for draw in range(N_BOOTSTRAP):
    sample = rng.integers(0, len(gap_by_zip), size=len(gap_by_zip))
    bootstrap_means[draw] = np.nanmean(gap_by_zip.to_numpy()[sample], axis=0)
event_study["GapLower"] = np.nanpercentile(bootstrap_means, 2.5, axis=0)
event_study["GapUpper"] = np.nanpercentile(bootstrap_means, 97.5, axis=0)

print(event_study.loc[[-24, -12, 0, 6, 12, 24]].round(4).to_string())

In [ ]:
def log_to_pct(log_change):
    return 100 * (np.exp(log_change) - 1)

TREATED_COLOR, CONTROL_COLOR = "#2a78d6", "#eb6834"
INK, MUTED, GRID, BASELINE = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7"

fig, (path_axis, gap_axis) = plt.subplots(
    2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": [3, 2], "hspace": 0.12}
)
event_time = event_study.index

# Top: the average treated zip's path against its matched controls, both indexed to 0 at
# the anchor month.
path_axis.plot(event_time, log_to_pct(event_study["Treated"]), color=TREATED_COLOR, linewidth=2, label="Data center zip codes")
path_axis.plot(event_time, log_to_pct(event_study["Control"]), color=CONTROL_COLOR, linewidth=2, label="Matched control zip codes")
for series, color in [("Treated", TREATED_COLOR), ("Control", CONTROL_COLOR)]:
    path_axis.annotate(
        series.lower(), (event_time[-1], log_to_pct(event_study[series].iloc[-1])),
        xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color=INK,
    )
path_axis.set_ylabel("Home value change since anchor month (%)", color=INK)
path_axis.legend(frameon=False, loc="upper left")
path_axis.set_title(
    f"Home values around data center events ({EVENT_OF_INTEREST.name.replace('_', ' ').lower()})",
    loc="left", color=INK, fontsize=12,
)

# Bottom: the treated-minus-control gap with its bootstrap interval. The pre-period should
# hug zero; that is the parallel-trends check.
gap_axis.fill_between(
    event_time, log_to_pct(event_study["GapLower"]), log_to_pct(event_study["GapUpper"]),
    color=TREATED_COLOR, alpha=0.12, linewidth=0, label="95% bootstrap interval",
)
gap_axis.plot(event_time, log_to_pct(event_study["Gap"]), color=TREATED_COLOR, linewidth=2, label="Treated minus control")
gap_axis.axhline(0, color=BASELINE, linewidth=1)
gap_axis.set_ylabel("Gap (percentage points)", color=INK)
gap_axis.set_xlabel("Months relative to anchor month", color=INK)
gap_axis.legend(frameon=False, loc="lower left")
# Treated zips observed at each point; the count falls off after the anchor as recent events
# run out of data.
for month in range(-EVENT_WINDOW_MONTHS, EVENT_WINDOW_MONTHS + 1, 6):
    gap_axis.annotate(
        f"n={event_study.loc[month, 'N']}", (month, 1), xytext=(0, -4), textcoords="offset points",
        xycoords=("data", "axes fraction"), ha="center", va="top", fontsize=8, color=MUTED,
    )

for axis in (path_axis, gap_axis):
    axis.axvline(0, color=BASELINE, linewidth=1)
    axis.grid(axis="y", color=GRID, linewidth=1)
    axis.tick_params(colors=MUTED, labelcolor=INK)
    for side in ("top", "right"):
        axis.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        axis.spines[side].set_color(BASELINE)
gap_axis.set_xticks(range(-EVENT_WINDOW_MONTHS, EVENT_WINDOW_MONTHS + 1, 6))
plt.show()

In [ ]:
HORIZONS_MONTHS = [3, 6, 12]

# The headline numbers: the gap at each fixed horizon after the anchor, in percent, with
# its interval and the number of treated zips observed that far out.
horizon_estimates = event_study.loc[HORIZONS_MONTHS, ["N", "Gap", "GapLower", "GapUpper"]].copy()
for column in ["Gap", "GapLower", "GapUpper"]:
    horizon_estimates[f"{column}Pct"] = log_to_pct(horizon_estimates.pop(column))
horizon_estimates.index.name = "MonthsAfterAnchor"

print(horizon_estimates.round(2).to_string())